<center> <b> Dataset n Dataloader Custom

Dataset is one of the fundamental things in ai, cuz almost all pipeline training have the same flow like this:

```
Raw Data
   ↓
Dataset
   ↓
DataLoader
   ↓
Batch Tensor
   ↓
Model
```

Dataset in simple term is the way u take one sample data

In [10]:
import torch
from torch.utils.data import Dataset,DataLoader

In [2]:
# basic skeleton dataset
class MyDataset(Dataset):

    def __init__(self):
        pass

    def __len__(self):
        pass

    def __getitem__(self, idx):
        pass

# \__len__()

it return numbers of sample, torch need for:
- batching
- (if) make progress bar
- iteration
- shuffling

In [3]:
def __len__(self):
    return len(self.data)

# \__getitem__(idx)

it take one sample

In [4]:
def __getitem__(self,idx):
    x = self.data[idx]
    y = self.labels[idx]

    return x, y

if we take dataset[5] => so getitem take one sample data in index-5 and then return the x,y

# Simple case

In [12]:
class NumberDataset(Dataset):

    def __init__(self):
        self.data = [10,21,34,48,56,100,43,226]

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):

        x = self.data[idx]

        return torch.tensor(x)
    
dataset = NumberDataset()

In [13]:
print(len(dataset))
print(dataset[0]) # <- first index of tensors

8
tensor(10)


# Dataloader

if dataset take it one sample, dataloader here to make batching process, shuffle and paralell loading

In [14]:
loader = DataLoader(
    dataset,
    batch_size=2,
    shuffle=True
)

for batch in loader:
    print(batch)

# u see that the tensor sperated by dataloader into num of group that we set in batch size 
# dataloader also auto randomized the sequence due to shuffle=True effect 

tensor([34, 21])
tensor([56, 10])
tensor([ 48, 226])
tensor([100,  43])


# NLP CASE

let say we have data (happy or no)
| text              | label |
| ----------------- | ----- |
| i love ma mom so much ❤️ | 1     |
| the smile vivid from him in this accident was so sus   | 0     |


## Build tokenizer (commonly using hf transformers)

In [15]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

## Build custom dataset

In [17]:
class SentimentDataset(Dataset):
    def __init__(self,texts,labels,tokenizer,max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self,idx):
        # REMEMBER, take one sample
        text = self.texts[idx]
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            padding = 'max_length',
            truncation = True,
            max_length = self.max_len,
            return_tensors = 'pt'
        )

        return {
            'input_ids':encoding['input_ids'].squezee(0),
            'attention_mask':encoding['attention_mask'].squezee(0),
            'labels':torch.tensor(label)
        }


`Bit explanation`

- why padding='max_length'?
    - all the tensors must be in same shape, u see in our text, the 1st text have 7 token while 2nd text have 11 token
- why truncation=True?
    - if the input text too long, we must truncated it so that we can avoid error memory and mismatch shape
    - let say max_len only 128 and we have text 300 token, the 272 token after it will be truncated 
- why return_tensors='pt?
    - if we dont set, the output is list. So we set to directly in form of tensors output
- why we squezee?
    - cuz of tokenizer we do before, the shape return [1,128], considered batch size=1 tough dataset must be return single sample:
    - [1,128] -> squezee(0) = [128]

# COMPVIS CASE

let say we've folder like:
```dataset/

    cat/
        1.jpg
        2.jpg

    cow/
        3.jpg
        4.jpg```

In [18]:
from PIL import Image
from torch.utils.data import Dataset
import torchvision.transforms as transforms

## Build custom dataset

In [ ]:
class CatCowDataset(Dataset):
    def __init__(self,image_paths,labels):
        self.image_paths = image_paths
        self.labels = labels

        self.transform = transforms.compose([
            transforms.resize((224,224)),
            transforms.RandomRotation(10),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self,idx):
        image_path = self.image_paths[idx]
        image = Image.open(image_path).convert('RGB')
        image = self.transform(image)

        label = self.labels[idx]

        return image,torch.tensor(label)

- transform crucial there bcuz model need to consistent shape, tensor
- with augmentation e.g. rotation, flip, solarize etc =>  model will more robust

# Bonus ✨✨✨